[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coopster-seclusion/sar-optical-pipeline/blob/main/notebooks/06_statistical_outputs.ipynb)

# Notebook 06: Statistical Outputs

Loads pre-computed change masks and delta stacks from notebook 05, then produces
four sets of statistical outputs without re-running any change detection.

All epoch lists, baseline year, polarizations, sigma multiplier, site name, and
AOI paths come from `config.yaml` — no hardcoded values in this notebook.

**Prerequisites:** Run notebook 05 (`05_change_detection.ipynb`) first.

**Inputs consumed:**
- `data/stats/05_sigma_thresholds.json` — sigma and threshold values
- `data/figures/{sensor}_{pol}_mask_{year}.tif` — per-epoch binary change masks
- `data/processed/{sensor}_{pol}_delta.nc` — log-ratio delta stacks
- `data/stacks/{sensor}_{pol}.nc` — full dB backscatter stacks (for timeseries)

**Outputs written to `outputs/stats/`:**
- `06_{sensor}_{pol}_change_stats.csv` — per-epoch area, pixel count, delta statistics
- `06_{sensor}_{pol}_zonal_timeseries.csv` — per-epoch mean backscatter by AOI polygon
- `06_{sensor}_{pol}_composite_change.tif` — union of all post-baseline change masks
- `06_summary.json` — machine-readable summary of the full run

## Setup

In [ ]:
# ── Cell 1: run this first, every session ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = 'https://github.com/coopster-seclusion/sar-optical-pipeline.git'
REPO_DIR = '/content/sar-optical-pipeline'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'rioxarray', 'netcdf4', 'pyogrio', 'rasterio'], check=True)

import json
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from pathlib import Path

from pipeline.env import setup, repo_root
from pipeline.validate import validate_config
from pipeline.utils import resolve_crs, save_raster
import pipeline.change as change_lib

cfg, DATA, OUT = setup(colab=True)
validate_config(cfg, repo_root())

SITE        = cfg['site_name']
ROOT        = repo_root()
OUT_STATS   = OUT / 'stats'
OUT_STATS.mkdir(parents=True, exist_ok=True)
STACKS_DIR  = DATA / 'stacks'

aoi_path = ROOT / cfg['aoi']['change_area']
ref_path = ROOT / cfg['aoi']['stable_reference']
crs      = resolve_crs(cfg, aoi_path)

all_years           = [e['year'] for e in cfg['epochs']]
baseline            = cfg['change_detection']['baseline_year']
multiplier          = float(cfg['change_detection']['sigma_threshold_multiplier'])
pols                = cfg['opera'].get('polarizations') or ['HH', 'HV']
post_baseline_years = [y for y in all_years if y != baseline]

print(f'Site              : {SITE}')
print(f'Baseline year     : {baseline}')
print(f'Post-baseline yrs : {post_baseline_years}')
print(f'Polarizations     : {pols}')
print(f'Output CRS        : {crs}')
print(f'Stats output dir  : {OUT_STATS}')

## Load sigma thresholds from notebook 05

The JSON written by notebook 05 records the sigma values and active approach used
for change detection. This notebook uses the same active approach so that the change
stats are consistent with the masks already on disk.

To switch approaches (A vs B), change `active_sigmas` below and re-run.

In [ ]:
# ── Cell 2: load sigma thresholds ─────────────────────────────────────────────────────
sigma_json_path = DATA / 'stats' / '05_sigma_thresholds.json'

if not sigma_json_path.exists():
    raise FileNotFoundError(
        f'Sigma thresholds not found: {sigma_json_path}\n'
        'Run notebook 05 first to generate this file.'
    )

with open(sigma_json_path) as fh:
    sigma_report = json.load(fh)

# Mirrors the default active approach in notebook 05 (approach A: baseline-only).
# Change to 'approach_b_temporal' here if you re-ran notebook 05 with approach B.
active_sigmas = sigma_report.get('approach_a_baseline', {})

print(f'Loaded : {sigma_json_path.name}')
print(f'Site   : {sigma_report.get("site")}')
print(f'Baseline year in report: {sigma_report.get("baseline_year")}')
print()
for key, sigma in active_sigmas.items():
    print(f'  {key:<15}  sigma = {sigma:.3f} dB   threshold = {sigma * multiplier:.2f} dB')

## Load change masks from GeoTIFFs

Reads the binary change mask GeoTIFFs written by notebook 05 — one file per
sensor × polarization × epoch.  Missing files are skipped with a warning; at
least one sensor must have masks or an error is raised.

`1` = change detected, `0` = no change, `NaN` = nodata.

In [ ]:
# ── Cell 3: load per-epoch change masks ───────────────────────────────────────────────
# Pattern: data/figures/{sensor}_{pol}_mask_{year}.tif
masks_opera = {}   # pol → {year: DataArray(y, x)}
masks_hyp3  = {}

for sensor, masks_dict in [('opera', masks_opera), ('hyp3', masks_hyp3)]:
    for pol in pols:
        epoch_masks = {}
        for year in all_years:
            mask_path = DATA / 'figures' / f'{sensor}_{pol}_mask_{year}.tif'
            if mask_path.exists():
                da = rxr.open_rasterio(mask_path, masked=True).squeeze('band', drop=True)
                epoch_masks[year] = da
                print(f'Loaded  {sensor.upper()} {pol} {year}: shape={da.shape}')
            else:
                print(f'Missing {sensor.upper()} {pol} {year}: {mask_path.name}')
        if epoch_masks:
            masks_dict[pol] = epoch_masks

if not masks_opera and not masks_hyp3:
    raise RuntimeError(
        'No change masks found in data/figures/. '
        'Run notebook 05 first to generate the change mask GeoTIFFs.'
    )

## Load delta stacks from NetCDF

The delta stacks written by notebook 05 (`data/processed/{sensor}_{pol}_delta.nc`)
are used to compute mean and standard deviation of the change signal within flagged
pixels.  They are not recomputed here.

In [ ]:
# ── Cell 4: load delta stacks ─────────────────────────────────────────────────────────
# Pattern: data/processed/{sensor}_{pol}_delta.nc
delta_opera = {}   # pol → DataArray(time, y, x)
delta_hyp3  = {}

for pol in pols:
    opera_path = DATA / 'processed' / f'opera_{pol}_delta.nc'
    hyp3_path  = DATA / 'processed' / f'hyp3_{pol}_delta.nc'

    if opera_path.exists():
        delta_opera[pol] = xr.open_dataarray(opera_path, engine='netcdf4')
        yrs = [pd.Timestamp(t).year for t in delta_opera[pol].time.values]
        print(f'OPERA delta {pol}: shape={delta_opera[pol].shape}  years={yrs}')
    else:
        print(f'OPERA delta {pol}: NOT FOUND — {opera_path.name}')

    if hyp3_path.exists():
        delta_hyp3[pol] = xr.open_dataarray(hyp3_path, engine='netcdf4')
        yrs = [pd.Timestamp(t).year for t in delta_hyp3[pol].time.values]
        print(f'HyP3  delta {pol}: shape={delta_hyp3[pol].shape}  years={yrs}')
    else:
        print(f'HyP3  delta {pol}: NOT FOUND — {hyp3_path.name}')

## AOI area and pixel resolution

AOI area is computed from the change-area GeoJSON reprojected to the analysis CRS
(not hardcoded).  Pixel area is read from the Affine transform of the first available
mask GeoTIFF (not hardcoded).  Both values are used as denominators in the change stats.

In [ ]:
# ── Cell 5: AOI area and pixel resolution ─────────────────────────────────────────────
aoi_gdf      = gpd.read_file(aoi_path).to_crs(crs)
aoi_area_m2  = float(aoi_gdf.union_all().area)
aoi_area_km2 = aoi_area_m2 / 1e6
print(f'AOI area: {aoi_area_km2:.4f} km²  ({aoi_area_m2:,.0f} m²)')
print()

# Pixel area — derived from the Affine transform of the first available mask
pixel_areas = {}   # 'sensor_pol' → pixel area in km²
for sensor, masks_dict in [('opera', masks_opera), ('hyp3', masks_hyp3)]:
    for pol, epoch_masks in masks_dict.items():
        first_da = next(iter(epoch_masks.values()))
        res      = first_da.rio.resolution()          # (x_res, y_res); y_res is negative
        px_km2   = abs(res[0]) * abs(res[1]) / 1e6
        pixel_areas[f'{sensor}_{pol}'] = px_km2
        print(f'{sensor.upper()} {pol}: pixel = {abs(res[0]):.1f} m x {abs(res[1]):.1f} m  '
              f'-> pixel area = {px_km2 * 1e6:.0f} m2')

## Per-epoch change-area statistics

For each post-baseline epoch, counts changed pixels from the mask and computes:
- **area_changed_km2** = n_changed_pixels × pixel_area
- **pct_changed** = area_changed / AOI_area × 100
- **mean_delta_db / std_delta_db** = statistics of the delta within changed pixels

One CSV per sensor × polarization is written to `outputs/stats/`.

In [ ]:
# ── Cell 6: per-epoch change-area statistics ──────────────────────────────────────────
def _epoch_change_stats(
    sensor, pol, epoch_masks, delta_da,
    active_sigmas, multiplier, aoi_area_km2, pixel_area_km2, baseline,
):
    key       = f'{sensor}_{pol}'
    sigma     = active_sigmas.get(key)
    threshold = (sigma * multiplier) if sigma is not None else float('nan')
    rows = []
    for year in sorted(epoch_masks):
        if year == baseline:
            continue
        mask_vals = epoch_masks[year].values           # (y, x): 1=change, 0=none, NaN=nodata
        n_changed = int(np.nansum(mask_vals == 1))
        area_km2  = n_changed * pixel_area_km2
        pct       = (area_km2 / aoi_area_km2 * 100) if aoi_area_km2 > 0 else float('nan')

        mean_delta = std_delta = float('nan')
        if delta_da is not None:
            try:
                d_vals  = delta_da.sel(time=str(year), method='nearest').values
                changed = d_vals[mask_vals == 1]
                finite  = changed[np.isfinite(changed)]
                if finite.size > 0:
                    mean_delta = float(np.mean(finite))
                    std_delta  = float(np.std(finite))
            except Exception:
                pass

        rows.append({
            'year'            : year,
            'sensor'          : sensor,
            'pol'             : pol,
            'sigma'           : round(sigma, 4)     if sigma is not None        else float('nan'),
            'threshold_db'    : round(threshold, 4) if np.isfinite(threshold)   else float('nan'),
            'n_changed_pixels': n_changed,
            'area_changed_km2': round(area_km2, 4),
            'pct_changed'     : round(pct, 2),
            'mean_delta_db'   : round(mean_delta, 3) if np.isfinite(mean_delta) else float('nan'),
            'std_delta_db'    : round(std_delta,  3) if np.isfinite(std_delta)  else float('nan'),
        })
    return pd.DataFrame(rows)


all_stats_dfs = []

for sensor, masks_dict, delta_dict in [
    ('opera', masks_opera, delta_opera),
    ('hyp3',  masks_hyp3,  delta_hyp3),
]:
    for pol, epoch_masks in masks_dict.items():
        key            = f'{sensor}_{pol}'
        pixel_area_km2 = pixel_areas.get(key, float('nan'))
        delta_da       = delta_dict.get(pol)

        df = _epoch_change_stats(
            sensor, pol, epoch_masks, delta_da,
            active_sigmas, multiplier, aoi_area_km2, pixel_area_km2, baseline,
        )
        out_path = OUT_STATS / f'06_{key}_change_stats.csv'
        df.to_csv(out_path, index=False)
        all_stats_dfs.append(df)
        print(f'Saved: {out_path.name}')
        print(df.to_string(index=False))
        print()

if not all_stats_dfs:
    print('WARNING: no change stats computed — check that masks are present above.')

## Zonal backscatter timeseries

Calls `pipeline.change.zonal_stats_timeseries()` on the full dB backscatter stacks
(not the delta stacks) to compute per-epoch mean, std, and pixel count within the
change area and stable reference polygons.  One CSV per sensor × polarization.

In [ ]:
# ── Cell 7: zonal backscatter timeseries ──────────────────────────────────────────────
aoi_polygons = {
    'change_area'      : aoi_path,
    'stable_reference' : ref_path,
}

for sensor in ['opera', 'hyp3']:
    for pol in pols:
        stack_path = STACKS_DIR / f'{sensor}_{pol}.nc'
        if not stack_path.exists():
            print(f'{sensor.upper()} {pol} stack not found ({stack_path.name}) — skipping')
            continue

        da    = xr.open_dataarray(stack_path, engine='netcdf4')
        df_ts = change_lib.zonal_stats_timeseries(da, aoi_polygons, crs)
        df_ts['sensor'] = f'{sensor}_{pol}'

        out_path = OUT_STATS / f'06_{sensor}_{pol}_zonal_timeseries.csv'
        df_ts.to_csv(out_path, index=False)
        print(f'Saved: {out_path.name}')
        print(df_ts.to_string(index=False))
        print()

## Composite change map

For each sensor × polarization, takes the union across all post-baseline epoch masks:
- **1** — pixel was flagged as changed in at least one post-baseline epoch
- **0** — pixel had valid data in every epoch but was never flagged
- **NaN** — pixel had no valid data in any epoch (nodata / outside coverage)

Saved as a float32 GeoTIFF to `outputs/stats/`.

In [ ]:
# ── Cell 8: composite change map ──────────────────────────────────────────────────────
for sensor, masks_dict in [('opera', masks_opera), ('hyp3', masks_hyp3)]:
    for pol, epoch_masks in masks_dict.items():
        post_masks = {y: da for y, da in epoch_masks.items() if y != baseline}
        if not post_masks:
            print(f'{sensor.upper()} {pol}: no post-baseline masks — skipping composite')
            continue

        first_da  = next(iter(epoch_masks.values()))   # any mask for spatial reference

        # Stack post-baseline masks along a new axis: (n_epochs, y, x)
        arrays_np = np.stack(
            [post_masks[y].values for y in sorted(post_masks)], axis=0
        ).astype(np.float32)

        any_valid   = np.any(np.isfinite(arrays_np), axis=0)   # (y, x)
        any_changed = np.any(arrays_np == 1, axis=0)            # (y, x)

        # 1 where any epoch changed; 0 where none; NaN where no epoch had valid data
        composite_vals = np.where(
            ~any_valid, np.nan, np.where(any_changed, 1.0, 0.0)
        ).astype(np.float32)

        composite = xr.DataArray(
            composite_vals,
            dims=['y', 'x'],
            coords={'y': first_da.y.values, 'x': first_da.x.values},
        )
        composite = composite.rio.write_crs(crs)

        out_path = OUT_STATS / f'06_{sensor}_{pol}_composite_change.tif'
        save_raster(composite, out_path, dtype='float32', nodata=float('nan'))

        n_flagged  = int(np.nansum(composite_vals == 1))
        px_km2     = pixel_areas.get(f'{sensor}_{pol}', 0.0)
        area_flagged = n_flagged * px_km2
        pct_flagged  = (area_flagged / aoi_area_km2 * 100) if aoi_area_km2 > 0 else float('nan')

        print(f'Saved: {out_path.name}')
        print(f'  Flagged in >=1 epoch: {n_flagged} px  '
              f'{area_flagged:.4f} km2  ~{pct_flagged:.1f}% of AOI')

## Summary JSON

Writes a machine-readable summary of the full notebook 06 run to
`outputs/stats/06_summary.json`.  Contains site metadata and per-sensor change
statistics suitable for downstream reporting or ingest into a dashboard.

In [ ]:
# ── Cell 9: summary JSON ──────────────────────────────────────────────────────────────
def _nan_to_none(lst):
    """Convert NaN floats to None so the JSON is spec-compliant."""
    return [None if (isinstance(v, float) and not np.isfinite(v)) else v for v in lst]


sensors_present = []
if masks_opera:
    sensors_present.append('opera')
if masks_hyp3:
    sensors_present.append('hyp3')

per_sensor = {}
for df in all_stats_dfs:
    if df.empty:
        continue
    key = f"{df['sensor'].iloc[0]}_{df['pol'].iloc[0]}"
    per_sensor[key] = {
        'epochs'           : df['year'].tolist(),
        'area_changed_km2' : _nan_to_none(df['area_changed_km2'].tolist()),
        'pct_changed'      : _nan_to_none(df['pct_changed'].tolist()),
        'mean_delta_db'    : _nan_to_none(df['mean_delta_db'].tolist()),
        'std_delta_db'     : _nan_to_none(df['std_delta_db'].tolist()),
    }

summary = {
    'site_name'        : SITE,
    'baseline_year'    : baseline,
    'epochs_processed' : post_baseline_years,
    'sensors_present'  : sensors_present,
    'total_area_km2'   : round(aoi_area_km2, 4),
    'per_sensor'       : per_sensor,
}

summary_path = OUT_STATS / '06_summary.json'
with open(summary_path, 'w') as fh:
    json.dump(summary, fh, indent=2)

print(f'Saved: {summary_path.name}')
print()
print(json.dumps(summary, indent=2))

In [ ]:
# ── Cell 10: output file listing ──────────────────────────────────────────────────────
print('=' * 60)
print(f'Notebook 06 complete  —  {SITE}')
print('=' * 60)
print(f'\nOutputs: {OUT_STATS}\n')
for f in sorted(OUT_STATS.glob('06_*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<55}  {size_kb:.1f} KB')